# Blackjack Strategy Comparison
## A Monte Carlo Simulation Study

How much does *how you play* actually change what happens at a blackjack table? This notebook answers that with four strategies played over **one million hands each** — four million hands total, ~5.8 million recorded decisions — under a single fixed rule set.

The simulator is built from scratch: a pluggable strategy interface (any decision logic implements one `decide()` method), configurable casino rules, per-decision recording with full game context, and a fixed seed for reproducibility. Rules throughout: **6 decks, dealer stands on soft 17, blackjack pays 3:2, double on any two cards, no surrender.**

| Strategy | What it knows |
|---|---|
| **Basic** | Mathematically optimal play — the full hit / stand / double / split table |
| **Semi-Random** | Two rules only: never stand when busting is impossible; stand on hard 17+. Random otherwise |
| **Dealer Mirror** | Plays by the dealer's own rules — hit to 17, then stand. Never doubles or splits |
| **Random** | Uniformly random legal action — the pure baseline |

The goal isn't only to rank them. It's to find *where* the edge is won and lost — the specific decisions that separate optimal play from naive heuristics, and the common intuitions about blackjack that turn out to be wrong.

*Session-level analysis — bankrolls, betting systems, card counting, risk of ruin — lives in `session_analysis.ipynb`.*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titleweight": "bold"})

RUNS_DIR = "../data/runs"
BASE_BET = 10

STRATEGY_MAP = {
    "Basic":         "hands_basic",
    "Semi-Random":   "hands_semi_random",
    "Dealer Mirror": "hands_dealer_mirror",
    "Random":        "hands_random",
}
STRAT_COLORS = {"Basic": "#2196F3", "Semi-Random": "#4CAF50",
                "Dealer Mirror": "#FF9800", "Random": "#F44336"}

# Load only the columns the analysis needs -> far less memory/IO on the big logs.
USE_COLS = ["hand_id", "decision_index", "player_value", "player_is_soft",
            "dealer_upcard", "action", "final_dealer_value", "outcome", "is_win", "payout"]
DTYPES = {"player_value": "int16", "dealer_upcard": "int16", "decision_index": "int16",
          "is_win": "int8", "player_is_soft": "bool", "action": "category", "outcome": "category"}

missing = [s for s, r in STRATEGY_MAP.items()
           if not os.path.exists(f"{RUNS_DIR}/{r}/run_metadata.json")]
if missing:
    raise FileNotFoundError(
        f"Missing runs: {missing}. Re-run regenerate_data.py, then RESTART the kernel "
        f"and Run All (loading while the sim is still writing leaves the data incomplete).")

decisions, hands = {}, {}
for name, run in STRATEGY_MAP.items():
    df = pd.read_csv(f"{RUNS_DIR}/{run}/decisions.csv", usecols=USE_COLS, dtype=DTYPES)
    decisions[name] = df
    hands[name] = df.drop_duplicates("hand_id", keep="first")

print(f"{'Strategy':<16}{'Hands':>12}{'Decisions':>14}")
print("-" * 42)
for s in STRATEGY_MAP:
    print(f"{s:<16}{len(hands[s]):>12,}{len(decisions[s]):>14,}")

## Simulation Validation

Before trusting any result, check the numbers that *shouldn't* depend on strategy. Dealer bust rate, blackjack frequency, and the outcome totals are properties of the cards and the rules — if the engine is correct they stay roughly constant no matter how the player plays. If they drift with strategy, something is broken.

In [ ]:
def hand_metrics(h):
    n = len(h)
    return dict(
        hands=n,
        dealer_bust=(h["final_dealer_value"] > 21).mean() * 100,
        blackjack=(h["outcome"] == "blackjack").mean() * 100,
        win=(h["outcome"].isin(["win", "blackjack"])).mean() * 100,
        loss=(h["outcome"].isin(["lose", "bust"])).mean() * 100,
        push=(h["outcome"] == "push").mean() * 100,
        bust=(h["outcome"] == "bust").mean() * 100,
        house_edge=-h["payout"].sum() / (n * BASE_BET) * 100,
    )

metrics = pd.DataFrame({s: hand_metrics(h) for s, h in hands.items()}).T
metrics["outcome_sum"] = metrics[["win", "loss", "push"]].sum(axis=1)

# Styled validation table
cols = ["dealer_bust", "blackjack", "win", "loss", "push", "outcome_sum", "house_edge"]
headers = ["Strategy", "Dealer Bust", "Blackjack", "Total Win", "Total Loss", "Push", "Sum", "House Edge"]
cell_text = [[s] + [f"{metrics.loc[s, c]:.2f}%" for c in cols] for s in STRATEGY_MAP]

fig, ax = plt.subplots(figsize=(14, 3)); ax.axis("off")
tbl = ax.table(cellText=cell_text, colLabels=headers, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
for j in range(len(headers)):
    tbl[0, j].set_facecolor("#2C3E50"); tbl[0, j].set_text_props(color="white", fontweight="bold")
row_fill = {"Basic": "#D5F5E3", "Semi-Random": "#FEF9E7", "Dealer Mirror": "#FEF9E7", "Random": "#FDEDEC"}
for i, s in enumerate(STRATEGY_MAP):
    for j in range(len(headers)):
        tbl[i + 1, j].set_facecolor(row_fill[s])
        if j in (6, 7): tbl[i + 1, j].set_text_props(fontweight="bold")
plt.title("Simulation Validation — Strategy-Independent Invariants", fontsize=12, fontweight="bold", pad=15)
plt.tight_layout(); plt.show()

print("Dealer bust range : {:.2f}%-{:.2f}%".format(metrics.dealer_bust.min(), metrics.dealer_bust.max()))
print("Blackjack range   : {:.2f}%-{:.2f}%".format(metrics.blackjack.min(), metrics.blackjack.max()))
print("Outcome sums      : {}".format(sorted(metrics.outcome_sum.round(1).unique())))
print("Basic house edge  : {:.2f}%".format(metrics.loc["Basic", "house_edge"]))

Validation passes cleanly. Across all four strategies the **dealer busts 26.8–26.9%** of the time and **blackjacks land ~4.5%** of hands — flat regardless of player behaviour, exactly as the cards dictate. Every outcome distribution sums to 100%.

The anchor is Basic strategy's house edge: **0.45%**, squarely inside the known range for these rules. That one number says the engine reproduces real blackjack mathematics — so the strategy *differences* below are signal, not artefacts. The edge sits at the low end because *dealer stands on soft 17* is one of the most player-favourable common rules; the same game with H17 would run nearer 0.6%.

## Strategy Comparison: The Headlines

Four strategies, one million hands each. Three numbers carry the story: how often you win, how much the house keeps, and how often you bust.

In [ ]:
order = list(STRATEGY_MAP)
colors = [STRAT_COLORS[s] for s in order]
panels = [("win", "Total Win Rate", "Win Rate (%)", 60),
          ("house_edge", "House Edge", "House Edge (%)", 50),
          ("bust", "Bust Rate", "Bust Rate (%)", 40)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (col, title, ylab, ymax) in zip(axes, panels):
    vals = metrics.loc[order, col]
    ax.bar(order, vals, color=colors)
    ax.set_title(title); ax.set_ylabel(ylab); ax.set_ylim(0, ymax)
    for i, v in enumerate(vals):
        ax.text(i, v + ymax * 0.012, f"{v:.1f}%", ha="center", fontweight="bold")
plt.suptitle(f"Strategy Comparison — {len(hands['Basic']):,} Hands Each", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

### Key Findings

**Basic strategy is in a different league.** Its house edge of **0.45%** is about **45 points** better than random play's 45.8%. That gap is the entire value of knowing what you're doing — and almost all of it comes from a handful of decision types, not from playing every hand a little better.

**Win rate barely moves; the house edge moves enormously.** Basic wins 43.3% of hands, Random 30.0% — a 13-point spread. But the house-edge spread is 45 points. That disconnect is the first clue that *winning more hands* is not where the money is. What matters is what happens on the hands you lose, and how much is on the table when you're ahead.

**Bust rate is not a quality ranking.** Semi-Random busts *least* of all four (12.4%) — less than Basic (15.8%) — yet its edge is over 12× worse. Dealer Mirror busts nearly twice as often as Semi-Random (27.3%) at essentially the same edge. Bust rate plainly isn't tracking how good a strategy is. The next section explains why.

## How You Lose Matters: Bust vs. Standing Loss

A loss is a loss on the scoreboard, but not in what it teaches. Bust before the dealer acts and you never gave the dealer a chance to break. Lose at showdown and you played the hand out and got beaten. Splitting losses into those two kinds exposes the most counterintuitive result in the study.

In [ ]:
order = list(STRATEGY_MAP)
win  = metrics.loc[order, "win"]
push = metrics.loc[order, "push"]
stand_loss = metrics.loc[order, "loss"] - metrics.loc[order, "bust"]  # lose at showdown
bust = metrics.loc[order, "bust"]
x = np.arange(len(order))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].bar(x, win, 0.5, label="Win", color="#2196F3")
axes[0].bar(x, push, 0.5, bottom=win, label="Push", color="#9E9E9E")
axes[0].bar(x, stand_loss, 0.5, bottom=win + push, label="Loss (standing)", color="#FF9800")
axes[0].bar(x, bust, 0.5, bottom=win + push + stand_loss, label="Bust", color="#F44336")
axes[0].set_xticks(x); axes[0].set_xticklabels(order)
axes[0].set_ylabel("Percentage (%)"); axes[0].set_ylim(0, 105)
axes[0].set_title("Full Outcome Breakdown by Strategy")
axes[0].legend(loc="upper left", bbox_to_anchor=(0.01, 0.99))
for i in range(len(order)):
    axes[0].text(i, win.iloc[i] + push.iloc[i] + stand_loss.iloc[i] + bust.iloc[i] / 2,
                 f"{bust.iloc[i]:.1f}%", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    axes[0].text(i, win.iloc[i] + push.iloc[i] + stand_loss.iloc[i] / 2,
                 f"{stand_loss.iloc[i]:.1f}%", ha="center", va="center", color="white", fontsize=9, fontweight="bold")

bust_share = bust / metrics.loc[order, "loss"] * 100  # bust as % of all losses
axes[1].bar(x, bust_share, color=colors)
axes[1].set_xticks(x); axes[1].set_xticklabels(order)
axes[1].set_title("Bust as % of Total Losses"); axes[1].set_ylabel("Bust / Total Loss (%)"); axes[1].set_ylim(0, 70)
for i, v in enumerate(bust_share):
    axes[1].text(i, v + 0.6, f"{v:.0f}%", ha="center", fontweight="bold")
plt.suptitle("How Each Strategy Loses", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

### The Counterintuitive Finding: Lower Bust Rate ≠ Better Strategy

Semi-Random has the **lowest bust rate of all four strategies (12.4%)** — lower than Basic's 15.8% — and yet its house edge (5.59%) is more than **12× worse** than Basic's 0.45%. The "don't bust" instinct, treated as a goal in itself, is actively misleading.

The breakdown shows the mechanism. Semi-Random often *stands* on stiff totals (12–16) where Basic would hit. Standing dodges the bust — so its bust counter is low — but a 13 against a dealer 10 loses about as often whether you bust trying to improve it or stand and let the dealer make a hand. The loss just changes columns: **bust 12.4% → standing loss 37.6%**, the highest standing-loss rate of any strategy. Same outcome, different label.

Bust as a share of *total* losses makes it explicit: **Basic 33%, Semi-Random 25%, Dealer Mirror 56%, Random 45%.** Dealer Mirror and Random lose mostly by busting — reckless aggression. Semi-Random loses mostly at showdown — accidental timidity. Basic sits between them, not by splitting the difference but because it takes a bust risk *only when expected value says the bust is the lesser evil.* On 16 vs dealer 10 it hits, knowing it will usually bust, because standing on 16 vs 10 loses even more often. The right question is never "how do I avoid busting?" — it's "which action loses me the least?"

## Decision Analysis: Where Strategy Actually Matters

The headlines say *that* Basic is better. The decision maps say *where*. Each cell below is a game state — the player's total against the dealer's upcard — coloured by how often that state wins, with Basic's chosen action overlaid.

In [ ]:
ACTION_CODE = {"hit": "H", "stand": "S", "double": "D", "split": "P", "surrender": "R", "none": "-"}

def first_decisions(name):
    d = decisions[name][decisions[name]["decision_index"] == 0].copy()
    d["action"] = d["action"].astype(str)
    return d

def win_action_pivots(d):
    win_piv = d.pivot_table("is_win", "player_value", "dealer_upcard", "mean")
    act_piv = d.pivot_table("action", "player_value", "dealer_upcard",
                            aggfunc=lambda x: x.mode().iat[0])
    return win_piv, act_piv

fb = first_decisions("Basic")
win_piv, act_piv = win_action_pivots(fb)

annot = win_piv.copy().astype(object)
for r in win_piv.index:
    for c in win_piv.columns:
        wr = win_piv.loc[r, c]
        if pd.isna(wr):
            annot.loc[r, c] = ""
        else:
            a = act_piv.loc[r, c] if (r in act_piv.index and c in act_piv.columns) else ""
            annot.loc[r, c] = f"{wr:.2f}\n{ACTION_CODE.get(a, '')}"

plt.figure(figsize=(14, 11))
sns.heatmap(win_piv, annot=annot, fmt="", cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=0.5, cbar_kws={"label": "Win Rate"}, annot_kws={"size": 9})
plt.title("Win Rate + Basic Strategy Action  (H=Hit  S=Stand  D=Double  P=Split)",
          fontsize=13, fontweight="bold")
plt.xlabel("Dealer Upcard"); plt.ylabel("Player Value")
plt.tight_layout(); plt.show()

### Reading the Win-Rate Heatmap

Read the colour first — it's the difficulty of the *situation*, before the player decides anything:

- **Dealer 2–6 (weak upcards):** noticeably greener everywhere — the dealer is likely to bust, so even mediocre player hands stay competitive.
- **Dealer 9–11 (strong upcards):** red columns — the dealer completes strong hands and the player is on the back foot.
- **Player 17–20:** green across the board; strong totals win whatever the dealer shows.
- **Player 12–16 vs dealer 7+:** the kill zone — red everywhere. No good option exists here, only a least-bad one.

Now read the action letters. Doubling (**D**) clusters exactly in the green-and-yellow rows 9–11 against weak dealers — Basic commits more money only when already favoured (11 vs 6 wins 64%). Standing (**S**) appears on 12–16 vs 2–6, cells that still lose more than half the time — but standing hands the bust risk to the dealer instead of taking it yourself. Hitting (**H**) takes over on 12–16 vs 7+, where standing is worse still. Basic isn't trying to win every hand; most of the board is unfavourable. It does damage control in the red and presses its bet in the green.

*One caveat for purists: each row pools hard totals, soft totals, and same-value pairs, and the letter shown is the single most common action in that cell. It's an honest summary of what Basic does on average — not a substitute for a full hard / soft / pairs strategy card.*

In [ ]:
order = list(STRATEGY_MAP)
deterministic = {"Basic", "Dealer Mirror"}
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
for ax, name in zip(axes.flatten(), order):
    d = first_decisions(name)
    win_piv = d.pivot_table("is_win", "player_value", "dealer_upcard", "mean")
    if name in deterministic:
        act_piv = d.pivot_table("action", "player_value", "dealer_upcard",
                                aggfunc=lambda x: x.mode().iat[0])
        ann = win_piv.copy().astype(object)
        for r in win_piv.index:
            for c in win_piv.columns:
                wr = win_piv.loc[r, c]
                if pd.isna(wr): ann.loc[r, c] = ""; continue
                a = act_piv.loc[r, c] if (r in act_piv.index and c in act_piv.columns) else ""
                ann.loc[r, c] = f"{wr:.2f}\n{ACTION_CODE.get(a, '')}"
        sns.heatmap(win_piv, annot=ann, fmt="", cmap="RdYlGn", vmin=0, vmax=1,
                    linewidths=0.3, ax=ax, cbar=False, annot_kws={"size": 7})
        sub = "H=Hit  S=Stand  D=Double  P=Split"
    else:
        sns.heatmap(win_piv, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
                    linewidths=0.3, ax=ax, cbar=False, annot_kws={"size": 7})
        sub = "win rate only — no fixed action"
    ax.set_title(f"{name}  ({sub})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Dealer Upcard"); ax.set_ylabel("Player Value")
plt.suptitle("Action Heatmap Comparison — All Four Strategies", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

### What the Comparison Reveals

The four maps share an identical colour field — proof that win rate is a property of the *game state*, not the strategy. What changes is only what each strategy *does* in each state.

**Basic** shows deliberate structure: D on 9–11, S on stiff hands against weak dealers, H against strong ones, splits on pairs. **Dealer Mirror** shows a clean horizontal cut — hit everything below 17, stand above, no doubles, no splits, context ignored. **Semi-Random** and **Random** show no action structure at all; no fixed decision exists, only the same underlying win-rate field.

The lesson: win rate is handed to you by the cards. What a strategy controls is whether it *captures* that win rate through correct action — and whether it multiplies its bet (double / split) in the cells where it's already ahead. Dealer Mirror and the random strategies leave that second lever completely unused.

In [ ]:
b = first_decisions("Basic")
m = first_decisions("Dealer Mirror")
bw = b.pivot_table("is_win", "player_value", "dealer_upcard", "mean")
mw = m.pivot_table("is_win", "player_value", "dealer_upcard", "mean")
ba = b.pivot_table("action", "player_value", "dealer_upcard", aggfunc=lambda x: x.mode().iat[0])
ma = m.pivot_table("action", "player_value", "dealer_upcard", aggfunc=lambda x: x.mode().iat[0])

idx = bw.index.intersection(mw.index); col = bw.columns.intersection(mw.columns)
diff = (bw.loc[idx, col] - mw.loc[idx, col])

same = pd.DataFrame(False, index=idx, columns=col)
for r in idx:
    for c in col:
        if r in ba.index and c in ba.columns and r in ma.index and c in ma.columns:
            same.loc[r, c] = (ba.loc[r, c] == ma.loc[r, c])
diff_div = diff.where(~same)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(diff, annot=True, fmt=".2f", cmap="RdYlGn", vmin=-0.15, vmax=0.15,
            linewidths=0.3, ax=axes[0], cbar_kws={"label": "Win rate diff (Basic - Mirror)"})
axes[0].set_title("Win-Rate Difference — all cells"); axes[0].set_xlabel("Dealer Upcard"); axes[0].set_ylabel("Player Value")
sns.heatmap(diff_div, annot=True, fmt=".2f", cmap="RdYlGn", vmin=-0.15, vmax=0.15,
            linewidths=0.3, ax=axes[1], cbar_kws={"label": "Win rate diff (Basic - Mirror)"})
axes[1].set_title("Win-Rate Difference — only where actions diverge"); axes[1].set_xlabel("Dealer Upcard"); axes[1].set_ylabel("Player Value")
plt.suptitle("Where Basic Strategy Gains Its Edge Over Dealer Mirror", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

### Quantifying the Edge: Basic vs. Dealer Mirror

To isolate where the edge actually comes from, subtract Dealer Mirror's win rate from Basic's, then mask out every cell where the two make the *same* decision. What's left is the edge, cell by cell.

It concentrates in two regions. **Stiff hands vs weak dealers (12–16 vs 2–6):** solid green, peaking at **+18 points** on 16 vs dealer 5 — the single richest seam. Basic stands and lets a likely-busting dealer self-destruct; Dealer Mirror hits and busts itself first. **The doubling band (9–11):** Basic doubles into favourable cells, Dealer Mirror only ever hits, and the gap stays positive across the row.

One honest caveat the map surfaces: a few split cells (a low pair vs dealer 7–9) show a small *negative* win-rate difference for Basic. That isn't a mistake — per-hand win rate isn't expected value. Splitting can lower your chance of winning a given hand while raising your dollar expectation, because it puts a second bet onto a favourable position. Win rate is the right lens for most of this board, but on splits the EV and the win rate genuinely diverge.

In [ ]:
headers = ["Strategy", "Win Rate", "Bust Rate", "Push Rate", "House Edge", "Doubles/Splits"]
rows = [[s, f"{metrics.loc[s,'win']:.1f}%", f"{metrics.loc[s,'bust']:.1f}%",
         f"{metrics.loc[s,'push']:.1f}%", f"{metrics.loc[s,'house_edge']:.2f}%",
         "Yes" if s == "Basic" else "No"] for s in STRATEGY_MAP]

fig, ax = plt.subplots(figsize=(12, 4)); ax.axis("off")
tbl = ax.table(cellText=rows, colLabels=headers, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2, 2.2)
for j in range(len(headers)):
    tbl[0, j].set_facecolor("#2C3E50"); tbl[0, j].set_text_props(color="white", fontweight="bold")
row_fill = {"Basic": "#D5F5E3", "Semi-Random": "#FEF9E7", "Dealer Mirror": "#FEF9E7", "Random": "#FDEDEC"}
for i, s in enumerate(STRATEGY_MAP):
    for j in range(len(headers)):
        tbl[i + 1, j].set_facecolor(row_fill[s])
        if j == 4: tbl[i + 1, j].set_text_props(fontweight="bold")
plt.title(f"Strategy Comparison Summary — {len(hands['Basic']):,} Hands Each", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout(); plt.show()

## Conclusions

**1. Strategy is worth ~45 points of house edge — concentrated, not spread.** Basic cuts the edge from random play's 45.8% to 0.45%, but the gain lives in specific decisions: stand-vs-hit on stiff hands, and doubling / splitting in favourable spots. On most hands every strategy plays roughly alike.

**2. Avoiding busts is not the objective; expected value is.** Semi-Random posts the lowest bust rate (12.4%) and nearly the worst edge (5.59%). It dodges busts by standing into showdown losses. Basic accepts a higher bust rate (15.8%) on purpose, because on hands like 16 vs 10 busting-while-trying loses less than standing-and-hoping.

**3. The decisive ground is 12–16 vs dealer 2–6.** This region holds the largest win-rate gap between Basic and naive play — up to 18 points on a single hand. Stand, and let the weak dealer bust; don't bust first.

**4. Doubling and splitting are the only offensive weapons — and naive strategies never fire them.** They appear exclusively in already-favourable cells. Skipping them leaves real money uncaptured on the hands you *do* win.

**5. The game is structurally against the player; strategy is damage control.** Most of the win-rate map is red — the player acts first and can bust before the dealer plays. No strategy escapes that. Basic minimises it. Card counting can repaint a few of those red cells — which is where `session_analysis.ipynb` picks up.

---

*Built on a from-scratch Monte Carlo simulator: pluggable strategies, configurable rules, per-decision recording, validated against known blackjack mathematics. In Phase 3 the same decision logs train a neural network to recover this strategy from data alone — without being told the rules.*